In [0]:
import json
from pyspark.sql.functions import * 
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.get("job_parameters")

In [0]:
parameters = json.loads(dbutils.widgets.get("job_parameters"))
csv_options = parameters.get("csv_options")
location = parameters.get("location")
raw_to_bronze_col_map = parameters.get("raw_to_bronze_col_map")
deduplication_cols = parameters.get("deduplication_cols")
job_id = parameters.get("job_id")
partition = parameters.get("partition")
target_table = parameters.get("catalog") + "." + parameters.get("target_table")
file_prefix = parameters.get("file_prefix")
trigger_jobs = parameters.get("trigger_jobs", [])

In [0]:
df = spark.read.options(**csv_options).csv(location+"/"+file_prefix)

In [0]:
schema_check_flag = df.columns == list(raw_to_bronze_col_map.keys())
if schema_check_flag == False:
    raise Exception("Schema check failed. Please check the schema of the incoming dataframe.")

In [0]:
enriched_df = df.select([col(c).alias(raw_to_bronze_col_map.get(c, c)) for c in df.columns]) \
        .withColumns({
            "file_name": expr("_metadata.file_path"),
            "creation_timestamp": lit(current_timestamp()),
            "updation_timestamp": lit(current_timestamp()),
            "partition": lit(f"{partition}"),
            "job_run": lit(f"{job_id}{partition}"),
            "parent_job_run": lit(f"{job_id}{partition}")
        })

In [0]:
enriched_df.write \
    .mode("append") \
    .saveAsTable(target_table)

In [0]:
operation_metrics = DeltaTable.forName(spark, target_table).history(1).select("operationMetrics.numOutputRows", "operationMetrics.numFiles", "operationMetrics.numOutputBytes")
display(operation_metrics)

In [0]:
# Trigger Next Jobs
for job in trigger_jobs:
    with open(f"/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/jobs/{job['job_type']}/{job['job_id']}.json", "r") as job_loc:
        job_params = json.load(job_loc)
        job_params['partition'] = partition
        dbutils.notebook.run("/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform/ETL/bronze_to_silver", 500, {"job_parameters" : json.dumps(job_params)})